# 🚀 Модуль 1. Урок 3. Сводные таблицы (Pivot Tables) и многомерные срезы

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_01_excel/lesson_03_pivot_tables_aggregations.ipynb)

## 📌 Бизнес-контекст и методология
Сводные таблицы позволяют за секунды сгруппировать миллионы строк по категориям и статусам.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

items_path = ensure_dataset("order_items.csv", subfolder="raw")
products_path = ensure_dataset("products.csv", subfolder="raw")
orders_path = ensure_dataset("orders.csv", subfolder="raw")

items = pd.read_csv(items_path)
products = pd.read_csv(products_path)
orders = pd.read_csv(orders_path)

df = items.merge(products, on="product_id").merge(orders, on="order_id")

pivot_report = pd.pivot_table(
    df,
    values='total_amount',
    index='category',
    columns='status',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Итого'
)
pivot_report.style.format('{:,.0f} руб.')


status,Cancelled,Completed,Refunded,Итого
category,,,,
Beauty,"46,977 руб.","602,967 руб.","47,737 руб.","697,681 руб."
Electronics,"47,449 руб.","740,145 руб.","74,825 руб.","862,419 руб."
Fashion,"42,665 руб.","719,947 руб.","49,721 руб.","812,333 руб."
Home & Kitchen,"26,043 руб.","547,300 руб.","52,688 руб.","626,032 руб."
Итого,"163,134 руб.","2,610,360 руб.","224,971 руб.","2,998,465 руб."


### 🎯 Практическое задание 1: Доля завершенной выручки по категориям
Постройте сводную таблицу и рассчитайте долю статуса `Completed` в общей сумме продаж категории.

In [3]:
### ВАШ КОД ЗДЕСЬ
cat_summary = df.groupby(['category', 'status'])['total_amount'].sum().unstack(fill_value=0)
cat_summary['total'] = cat_summary.sum(axis=1)
if 'Completed' in cat_summary.columns:
    cat_summary['completed_rate'] = cat_summary['Completed'] / cat_summary['total']
else:
    cat_summary['completed_rate'] = 1.0

print(cat_summary[['total', 'completed_rate']])

assert 'completed_rate' in cat_summary.columns, "Колонка completed_rate обязательна"
assert (cat_summary['completed_rate'] <= 1.0).all(), "Доля не может превышать 100%"
print("✅ Тест успешно пройден: конверсия рассчитана!")


status              total  completed_rate
category                                 
Beauty          697681.11        0.864245
Electronics     862418.83        0.858220
Fashion         812333.25        0.886271
Home & Kitchen  626031.82        0.874237
✅ Тест успешно пройден: конверсия рассчитана!


### 💼 Calculated Fields
Создавайте расчетные поля вида `= SUM(Revenue) / COUNT(Orders)` вместо усреднения средних значений.